**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/hiraoka-2025"], check=True)
    os.chdir("/content/encore/entries/hiraoka-2025")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Hiraoka & Jo (2025) on DANDI:000044: the other neurons

Paper: T. Hiraoka and H.-H. Jo, "Hierarchical organization of bursty trains in event
sequences", doi:10.1063/5.0296506, read from the arXiv preprint body under `paper/`.
Dataset: DANDI:000044 version 0.210812.1516, the version the paper cites (ref. 39); see the
header of `reproduce_hiraoka_2025.ipynb` for the choice of version and the claims table.
This notebook tests one claim, `fig2-other-neurons-similar`, which needs a fit of every
unit and is kept apart for its run time (about 50 minutes on 4 processes). Every unit of the
eight sessions is fitted, so that the claim's pyramidal sets can be shown beside the
inhibitory units and all units (the paper's representative unit is labelled inhibitory).

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1"); os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time, resource
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import pandas as pd
import powerlaw
import bursttree as bt

T0 = time.time()
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_rows", 300)
print("dandiset", bt.DANDISET, "version", bt.DANDI_VERSION)
print("powerlaw", powerlaw.__version__, "; adopted readings (unbounded exponent):", bt.ADOPTED_READINGS)

## Data check

The units tables of all eight sessions: counts by cell type, and spike trains that are not
sorted in time (the burst analysis sorts each train before taking IETs).

In [ ]:
assets = bt.list_assets()
trains, rows = [], []
for a in assets:
    ses = a["path"].split("ses-")[1].split("_")[0]
    U = bt.load_units(bt.open_nwb(a["url"]))
    uns = [int(U["id"][i]) for i, s in enumerate(U["spikes"]) if np.any(np.diff(s) < 0)]
    rows.append(dict(session=ses, units=len(U["id"]), excitatory=int((U["cell_type"] == "excitatory").sum()),
                     inhibitory=int((U["cell_type"] == "inhibitory").sum()), unsorted_unit_ids=uns))
    for i, s in enumerate(U["spikes"]):
        trains.append(dict(session=ses, row=i, id=int(U["id"][i]), cell_type=U["cell_type"][i], n=len(s),
                           unsorted=int(U["id"][i]) in uns, spikes_ms=s * 1e3))
print(pd.DataFrame(rows).to_string())
print("units:", len(trains), "; by cell type:", pd.Series([t["cell_type"] for t in trains]).value_counts().to_dict(),
      "; with unsorted spike times:", [(t["session"], t["id"], t["cell_type"]) for t in trains if t["unsorted"]])
for ct in ("excitatory", "inhibitory"):
    ns = [t["n"] for t in trains if t["cell_type"] == ct]
    print(f"spikes per {ct} unit: min", min(ns), "median", int(np.median(ns)), "max", max(ns))

## fig2-other-neurons-similar

> Other neurons of the recordings show patterns similar to the representative unit, with
> heavy-tailed burst-size and merging-number distributions across timescales.

Criterion: the set is (a) the pyramidal units of session Achilles-10252013, or (b) the pyramidal
units of every session. "Pyramidal" is read as the `cell_type` value `excitatory`, the only
label for principal cells in this version. Only units whose relevant range holds at least three
grid points are used, with all spikes of the session (O2) and each fit of O3, under both grids
of O4. A unit shows the pattern when the power law is supported (R > 0, p < 0.05) for burst sizes
at more than half of its relevant timescales and for merging numbers at more than half of its
pairs (ratios 3.3, 5.9 and 11). Reproduced when more than half of the units show the pattern
under one reading. The distribution of each unit's median beta is reported.

The adopted fit has no upper bound on the exponent (helper, `READINGS`). Beside the claim's sets,
the same summary is shown for the inhibitory units and for all units, of the representative
session and of every session, because the paper's own representative unit is labelled
inhibitory (a question to a reader on the claims file's "pyramidal"). The powerlaw 2.0.0 default,
with the exponent capped at 3, is a variant run on grid (a) alone in `other_neurons_cap3.ipynb`.

In [ ]:
t1 = time.time()
with ProcessPoolExecutor(max_workers=4) as ex:
    res = list(ex.map(bt.unit_all_patterns, [t["spikes_ms"] for t in trains], chunksize=2))
print(f"{len(trains)} units fitted (unbounded exponent, both grids) in {(time.time() - t1) / 60:.1f} min")
R = pd.DataFrame([dict(session=t["session"], id=t["id"], cell_type=t["cell_type"], n_spikes=t["n"], unsorted=t["unsorted"], **r)
                  for t, rr in zip(trains, res) for r in rr])

def summarise(d):
    used = d[d.used]
    return pd.Series(dict(units=len(d), used=len(used), pattern=int(used.pattern.sum()),
                          frac_pattern=round(used.pattern.mean(), 3) if len(used) else np.nan,
                          median_b_frac=round(used.b_frac.median(), 3), median_k_frac=round(used.k_frac.median(), 3),
                          beta_q25=round(used.median_beta.quantile(0.25), 3), beta_median=round(used.median_beta.median(), 3),
                          beta_q75=round(used.median_beta.quantile(0.75), 3)))

SETS = {"(a) pyramidal, Achilles-10252013": lambda d: d[(d.session == "Achilles-10252013") & (d.cell_type == "excitatory")],
        "(b) pyramidal, all sessions": lambda d: d[d.cell_type == "excitatory"],
        "variant: inhibitory, Achilles-10252013": lambda d: d[(d.session == "Achilles-10252013") & (d.cell_type == "inhibitory")],
        "variant: inhibitory, all sessions": lambda d: d[d.cell_type == "inhibitory"],
        "variant: all units, Achilles-10252013": lambda d: d[d.session == "Achilles-10252013"],
        "variant: all units, all sessions": lambda d: d}
for name, sel in SETS.items():
    print(f"\nset {name}:")
    print(sel(R).groupby(["grid", "reading"]).apply(summarise).to_string())
Rb = SETS["(b) pyramidal, all sessions"](R)
print("\nset (b) by session, fraction of used units showing the pattern:")
print(Rb[Rb.used].pivot_table(index="session", columns=["grid", "reading"], values="pattern", aggfunc="mean").round(3).to_string())
print("\nset (b) by session, median over units of the median beta:")
print(Rb[Rb.used].pivot_table(index="session", columns=["grid", "reading"], values="median_beta", aggfunc="median").round(3).to_string())
print("\nunits dropped (fewer than three grid points), by cell type:")
print(R[~R.used].groupby(["grid", "cell_type"]).id.count().to_string())
print("\nwithout the units with unsorted spike times, set (b):")
print(Rb[~Rb.unsorted].groupby(["grid", "reading"]).apply(summarise)[["used", "pattern", "frac_pattern"]].to_string())

**Result: reproduced, under the continuous fit from xmin = 1 alone.** With the exponent unbounded,
502 of 554 (grid (a)) and 502 of 562 (grid (b)) pyramidal units of all sessions show the pattern
under the continuous fit from xmin = 1 (0.906 and 0.893), and 107 and 106 of 120 in
Achilles-10252013 (0.892 and 0.883), so the criterion's "under one reading" is met for both sets.
No other reading meets it: in set (b) the continuous KS fit gives 0.449 and 0.454, the discrete
KS fit 0.233 and 0.240, and the discrete fit from xmin = 1 0.211 and 0.215; in set (a) they give
0.342 and 0.292, 0.117 and 0.083, and 0.017. The pass is fragile: the continuous fit from
xmin = 1 treats integer sizes as continuous down to 1, where Clauset et al. (ref. 36, Section 3)
find the continuous approximation poor, and its exponents (median over units of the median beta
1.744 to 1.778) are far from the paper's 2.35. The inhibitory units, the class the files give the
paper's own representative unit, show the pattern far more often: 92 and 87 of 128 under the
discrete KS fit (0.719 and 0.680) and 95 of 128 under the continuous one (0.742); over all 690
units the continuous KS fit gives 0.504 and 0.507. Under the discrete KS fit the median over
pyramidal units of the median beta is 2.471 and 2.459 (interquartile range 2.216 to 2.789 on grid
(a)), and 2.284 to 2.600 by session. 32 pyramidal units have fewer than three points on grid (a)
and are not used there. Leaving out the two pyramidal units with unsorted spike times changes no
fraction by more than 0.003.

### Burst sizes and merging numbers taken apart

The pattern needs both parts. Their separate rates show which part fails where a reading
falls short.

In [ ]:
used = R[R.used]
def parts(d):
    return d.assign(burst_ok=d.b_frac > 0.5, merge_ok=d.k_frac > 0.5).groupby(["grid", "reading"]).agg(
        used=("id", "size"), burst_ok=("burst_ok", "mean"), merge_ok=("merge_ok", "mean"), both=("pattern", "mean"))
for name, sel in SETS.items():
    print(f"\nset {name}:"); print(parts(sel(used)).round(3).to_string())
print(f"\nnotebook time {(time.time() - T0) / 60:.1f} min, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB (main process)")

**Result (parts).** With the exponent unbounded and the discrete KS fit, the burst-size part of the
pattern holds in 0.744 and 0.770 of the pyramidal units of set (b), and the merging-number part in
0.251 and 0.249: it is the merging numbers that fail. In set (a) the merging part holds in 0.117
and 0.083. In the inhibitory units of all sessions the parts hold in 0.906 and 0.977 (burst sizes)
and 0.781 and 0.695 (merging numbers).